In [126]:
import gzip
import pickle
import os
import json
import pandas as pd
import os 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, balanced_accuracy_score,
                              recall_score, f1_score)
from sklearn.metrics import confusion_matrix


In [127]:
def cargar_datos():

    df_train=pd.read_csv("../files/input/train_data.csv.zip")
    df_test=pd.read_csv("../files/input/test_data.csv.zip")
    return df_train, df_test

In [128]:
def limpieza(df):
    df=df.copy()
    df=df.rename(columns={"default payment next month":"default"})
    df=df.drop(columns=["ID"], errors="ignore")
    df=df.dropna()
    df=df[(df["EDUCATION"]>0) & (df["MARRIAGE"]>0)]
    df.loc[df["EDUCATION"]>=4, "EDUCATION"]=4
    
    return df

In [129]:
train, test=cargar_datos()
train=limpieza(train)
train.head(5)

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,310000,1,3,1,32,0,0,0,0,0,...,84373,57779,14163,8295,6000,4000,3000,1000,2000,0
1,10000,2,3,1,49,-1,-1,-2,-1,2,...,1690,1138,930,0,0,2828,0,182,0,1
2,50000,1,2,1,28,-1,-1,-1,0,-1,...,45975,1300,43987,0,46257,2200,1300,43987,1386,0
3,80000,2,3,1,52,2,2,3,3,3,...,40748,39816,40607,3700,1600,1600,0,1600,1600,1
4,270000,1,1,2,34,1,2,0,0,2,...,22448,15490,17343,0,4000,2000,0,2000,2000,0


In [130]:
train

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,310000,1,3,1,32,0,0,0,0,0,...,84373,57779,14163,8295,6000,4000,3000,1000,2000,0
1,10000,2,3,1,49,-1,-1,-2,-1,2,...,1690,1138,930,0,0,2828,0,182,0,1
2,50000,1,2,1,28,-1,-1,-1,0,-1,...,45975,1300,43987,0,46257,2200,1300,43987,1386,0
3,80000,2,3,1,52,2,2,3,3,3,...,40748,39816,40607,3700,1600,1600,0,1600,1600,1
4,270000,1,1,2,34,1,2,0,0,2,...,22448,15490,17343,0,4000,2000,0,2000,2000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20995,140000,2,2,1,27,2,-1,-1,-1,0,...,1580,804,728,752,800,1580,0,700,700,0
20996,130000,1,2,2,41,0,0,0,0,0,...,123107,42897,39378,4442,5200,5012,2500,5000,2000,0
20997,50000,1,3,2,23,0,0,0,0,0,...,28967,29829,30046,1973,1426,1001,1432,1062,997,0
20998,90000,2,3,2,25,0,0,0,0,0,...,5613,10113,10113,3000,3000,0,4500,0,3440,0


In [131]:
columnas_categoricas=["SEX","EDUCATION","MARRIAGE"]
columnas_numericas=[col for col in train.columns if col not in columnas_categoricas]
columnas_numericas

['LIMIT_BAL',
 'AGE',
 'PAY_0',
 'PAY_2',
 'PAY_3',
 'PAY_4',
 'PAY_5',
 'PAY_6',
 'BILL_AMT1',
 'BILL_AMT2',
 'BILL_AMT3',
 'BILL_AMT4',
 'BILL_AMT5',
 'BILL_AMT6',
 'PAY_AMT1',
 'PAY_AMT2',
 'PAY_AMT3',
 'PAY_AMT4',
 'PAY_AMT5',
 'PAY_AMT6',
 'default']

In [132]:
def separar_datos(base):
    base=base.copy()
    x=base
    y=base.pop("default")
    return x,y

In [133]:
def hacer_pipeline(estimador):
    columnas_categoricas=["SEX","EDUCATION","MARRIAGE"]

    preproceso=ColumnTransformer(
        transformers=[("ohe",OneHotEncoder(handle_unknown="ignore"),columnas_categoricas)
                       ],
                       remainder=MinMaxScaler()
    )

    selec_best=SelectKBest(score_func=f_classif)

    pipeline=Pipeline(
        steps=[
            ("transform",preproceso),
            ("seleccion",selec_best),
            ("estimador",estimador)
        ]
    )

    return pipeline

In [134]:
def make_grid_search(estimador,param_grid,cv=10):

    grid_search=GridSearchCV(
        estimator=estimador,
        param_grid=param_grid,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
        verbose=1
    )

    return grid_search

In [135]:
def train_logistic_regression(x_train,y_train,x_test,y_test):
    log_reg=LogisticRegression(random_state=42,solver="liblinear")

    pipeline=hacer_pipeline(log_reg)

    param_grid=({
        "seleccion__k": range(1,25),
        "estimador__penalty":["l1","l2"],
        "estimador__C":[0.01,0.1,1,10,100]
        })

    estimador=make_grid_search(
        estimador=pipeline,
        param_grid=param_grid,
        cv=10
    )

    estimador.fit(x_train,y_train)

    return estimador

In [136]:
def eval_metrics(
    y_train_true,
    y_test_true,
    y_train_pred,
    y_test_pred,
):

    from sklearn.metrics import accuracy_score, balanced_accuracy_score

    accuracy_train = round(accuracy_score(y_train_true, y_train_pred), 4)
    accuracy_test = round(accuracy_score(y_test_true, y_test_pred), 4)
    balanced_accuracy_train = round(
        balanced_accuracy_score(y_train_true, y_train_pred), 4
    )
    balanced_accuracy_test = round(balanced_accuracy_score(y_test_true, y_test_pred), 4)

    return (
        accuracy_train,
        accuracy_test,
        balanced_accuracy_train,
        balanced_accuracy_test,
    )

In [137]:
def calculate_metrics(model, X, y, dataset_type):

    y_pred = model.predict(X)
    
    return {
        'type': 'metrics',
        'dataset': dataset_type,
        'precision': precision_score(y, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'f1_score': f1_score(y, y_pred)
    }

In [138]:
def calculate_confusion_matrix(model, X, y, dataset_type):
    y_pred = model.predict(X)
    cm = confusion_matrix(y, y_pred)
    
    return {
        'type': 'cm_matrix',
        'dataset': dataset_type,
        'true_0': {
            "predicted_0": int(cm[0,0]), 
            "predicted_1": int(cm[0,1])
        },
        'true_1': {
            "predicted_0": int(cm[1,0]), 
            "predicted_1": int(cm[1,1])
        }
    }

In [139]:
def guardar_modelo(modelo):
    carpeta="../files/models"
    if not os.path.exists(carpeta):
        os.makedirs(carpeta)

    nombre = os.path.join(carpeta, "model.pkl.gz")
    with gzip.open(nombre, 'wb') as f:
        pickle.dump(modelo, f)

In [140]:
def guardar_metricas(model, x_train, y_train, x_test, y_test):

    m_train = calculate_metrics(model, x_train, y_train, 'train')
    m_test  = calculate_metrics(model, x_test, y_test, 'test')
    
    cm_train = calculate_confusion_matrix(model, x_train, y_train, 'train')
    cm_test  = calculate_confusion_matrix(model, x_test, y_test, 'test')
    
    resultados = [m_train, m_test, cm_train, cm_test]
    
    carpeta = "../files/output"
    if not os.path.exists(carpeta):
        os.makedirs(carpeta)
        
    ruta_archivo = os.path.join(carpeta, "metrics.json")
    
    with open(ruta_archivo, "w") as f:
        for item in resultados:
            f.write(json.dumps(item) + "\n")

In [141]:
base_train=cargar_datos()[0]
base_test=cargar_datos()[1]
base_train=limpieza(base_train)
base_test=limpieza(base_test)
x_train,y_train=separar_datos(base_train)
x_test,y_test=separar_datos(base_test)
estimator=train_logistic_regression(x_train,y_train,x_test,y_test)

guardar_modelo(estimator)
guardar_metricas(estimator,x_train,y_train,x_test,y_test)


Fitting 10 folds for each of 240 candidates, totalling 2400 fits


In [142]:
print(estimator.best_params_)

{'estimador__C': 0.1, 'estimador__penalty': 'l1', 'seleccion__k': 1}
